In [7]:
import cv2
import os
import numpy as np
from pathlib import Path
import math

In [8]:
def crop_inscribed_square(input_dir, output_dir, threshold_value=15):
    """
    Remove bordas pretas de imagens dermatoscópicas recortando 
    o maior quadrado interno à área da pele.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    valid_extensions = ('.png', '.jpg', '.jpeg')
    
    for filename in os.listdir(input_dir):
        if not filename.lower().endswith(valid_extensions):
            continue
            
        img_path = os.path.join(input_dir, filename)
        out_path = os.path.join(output_dir, filename)
        
        img = cv2.imread(img_path)
        if img is None:
            continue
            
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY)
        
        # Encontra os contornos
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            
            # Encontra o centro e o raio mínimo que engloba o círculo da pele
            (x, y), radius = cv2.minEnclosingCircle(largest_contour)
            center = (int(x), int(y))
            radius = int(radius)
            
            # Calcula o tamanho do lado do maior quadrado inscrito no círculo
            # Reduzimos o raio em 5% (0.95) para garantir que nenhuma borda preta escape
            side_length = int((radius * 0.95) * math.sqrt(2))
            half_side = side_length // 2
            
            # Calcula as coordenadas do recorte garantindo que não saiam da imagem
            x1 = max(0, center[0] - half_side)
            y1 = max(0, center[1] - half_side)
            x2 = min(img.shape[1], center[0] + half_side)
            y2 = min(img.shape[0], center[1] + half_side)
            
            cropped_img = img[y1:y2, x1:x2]
            
            # Só salva se o recorte for válido
            if cropped_img.size > 0:
                cv2.imwrite(out_path, cropped_img)
                print(f"Sucesso: {filename} | Novo tamanho: {cropped_img.shape[:2]}")
            else:
                cv2.imwrite(out_path, img)
        else:
            cv2.imwrite(out_path, img)

In [10]:
pasta_origem = "/datasets/PAD-UFES-26/anonymous-images"
pasta_destino = "/datasets/PAD-UFES-26/anonymous-images-derm-cropped"

crop_inscribed_square(pasta_origem, pasta_destino)

Sucesso: 46745_61635_0.png | Novo tamanho: (702, 702)
Sucesso: 75459_2890_0.png | Novo tamanho: (682, 682)
Sucesso: 47911_45399_0.png | Novo tamanho: (682, 682)
Sucesso: 72166_63794_0.png | Novo tamanho: (682, 682)
Sucesso: 18081_8861_1.png | Novo tamanho: (682, 682)
Sucesso: 17053_98489_0.png | Novo tamanho: (2152, 2152)
Sucesso: 97455_20314_0.png | Novo tamanho: (682, 682)
Sucesso: 74904_8056_0.png | Novo tamanho: (682, 682)
Sucesso: 30753_32354_0.png | Novo tamanho: (670, 670)
Sucesso: 25787_15408_0.png | Novo tamanho: (634, 634)
Sucesso: 74166_36314_2.png | Novo tamanho: (532, 532)
Sucesso: 62761_70003_1.png | Novo tamanho: (320, 320)
Sucesso: 67743_99382_0.png | Novo tamanho: (278, 278)
Sucesso: 31209_38060_0.png | Novo tamanho: (1574, 1574)
Sucesso: 63296_33459_1.png | Novo tamanho: (682, 682)
Sucesso: 14063_32426_1.png | Novo tamanho: (380, 380)
Sucesso: 31891_66676_0.png | Novo tamanho: (682, 682)
Sucesso: 27693_7180_0.png | Novo tamanho: (682, 682)
Sucesso: 57377_79568_0.png |